In [1]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn tqdm pandas huggingface_hub
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.4 MB/s eta 0:00:00


In [2]:
# =========================================
# Cell 2 — Hugging Face login (needed for gated Llama)
# =========================================
import os, getpass
from huggingface_hub import login

HF_TOKEN = getpass.getpass("HF token (hidden): ").strip()
assert HF_TOKEN, "HF token is required for meta-llama models"

login(token=HF_TOKEN)
print("✅ HF login done")


HF token (hidden): ··········
✅ HF login done


In [8]:
# =========================================
# Cell 3  Load FPB + get TRUE label order from ClassLabel
# =========================================
from datasets import load_dataset

FPB_REPO = "ChanceFocus/en-fpb"
SPLIT = "test"

ds = load_dataset(FPB_REPO, split=SPLIT)

print(ds)
print("Columns:", ds.column_names)

# Get true label names/order from dataset metadata
if "label" in ds.features and hasattr(ds.features["label"], "names"):
    DS_LABELS = [x.strip().lower() for x in ds.features["label"].names]
    print("✅ Dataset label order:", DS_LABELS)
else:
    # fallback if not ClassLabel (rare)
    DS_LABELS = ["negative", "neutral", "positive"]
    print("⚠️ No ClassLabel found. Using fallback:", DS_LABELS)

print("Example row:", ds[0])


Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
⚠️ No ClassLabel found. Using fallback: ['negative', 'neutral', 'positive']
Example row: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


In [9]:
# =========================================
# Cell 4 — Load Llama-3.1-8B-Instruct (4-bit)
# =========================================
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ.get("HF_TOKEN", None), use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=os.environ.get("HF_TOKEN", None),
    device_map="auto",
    quantization_config=bnb,
)
model.eval()

print("✅ Model loaded:", MODEL_NAME)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded: meta-llama/Llama-3.1-8B-Instruct


In [10]:
# =========================================
# Cell 5  Eval loop using dataset label order (DS_LABELS)
# =========================================
import os, json, re, random
from tqdm import tqdm

def get_text(ex: dict) -> str:
    for k in ["text", "sentence", "content", "input"]:
        if k in ex and ex[k] is not None:
            return str(ex[k])
    raise KeyError(f"No text field found. Keys={list(ex.keys())}")

def get_gold(ex: dict) -> int:
    # Expect integer label for ClassLabel datasets
    for k in ["label", "gold", "sentiment"]:
        if k in ex and ex[k] is not None:
            v = ex[k]
            if isinstance(v, str):
                v2 = v.strip().lower()
                if v2 in DS_LABELS:
                    return DS_LABELS.index(v2)
                raise ValueError(f"Unknown label string: {v}")
            return int(v)
    raise KeyError(f"No label field found. Keys={list(ex.keys())}")

def parse_pred(raw: str):
    if not raw:
        return None
    s = raw.strip().lower()

    # exact/word match against dataset labels
    for lab in DS_LABELS:
        if s == lab:
            return lab
        if re.search(rf"\b{re.escape(lab)}\b", s):
            return lab

    # common aliases
    alias = {"pos":"positive","neg":"negative","neu":"neutral"}
    for a, full in alias.items():
        if re.search(rf"\b{a}\b", s) and full in DS_LABELS:
            return full
    return None

def llama_classify(text: str, max_new_tokens=8) -> str:
    # IMPORTANT: Use DS_LABELS in the prompt so model matches dataset mapping
    messages = [
        {"role": "system", "content": "You are a financial sentiment classifier."},
        {"role": "user", "content": (
            "Classify the sentiment of the following financial text.\n"
            f"Respond with exactly ONE label from: {', '.join(DS_LABELS)}.\n\n"
            f"Text: {text}\n"
            "Label:"
        )}
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        out = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen = out[0, input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

# ---- run config ----
SEED = 42
MAX_SAMPLES = None   # None = full split
SHUFFLE = True

random.seed(SEED)
os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_id = f"llama31_8b_fpb_{FPB_REPO.replace('/','_')}_{SPLIT}_DSLABELS"
raw_path = f"text_responses/{run_id}.jsonl"

# resume
done = set()
if os.path.exists(raw_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["row_index"])
            except:
                pass
print("Already completed:", len(done))

idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

with open(raw_path, "a", encoding="utf-8") as f:
    for i in tqdm(idxs, desc="Evaluating"):
        if i in done:
            continue

        ex = ds[i]
        text = get_text(ex)
        gold = get_gold(ex)

        raw = llama_classify(text)
        pred_label = parse_pred(raw)
        pred = DS_LABELS.index(pred_label) if pred_label in DS_LABELS else -1

        rec = {
            "row_index": i,
            "text": text,
            "gold": gold,
            "predicted_index": pred,
            "predicted_label": pred_label,
            "raw": raw,
            "correct": (pred == gold),
            "ds_labels": DS_LABELS,
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("✅ Saved:", raw_path)


Already completed: 0


Evaluating: 100%|██████████| 970/970 [03:31<00:00,  4.59it/s]

✅ Saved: text_responses/llama31_8b_fpb_ChanceFocus_en-fpb_test_DSLABELS.jsonl


In [11]:
# =========================================
# Cell 6 (REPLACE) — Metrics (pure Python) for the new JSONL
# =========================================
import json, os

rows = []
with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            rows.append(json.loads(line))
        except:
            pass

gold = [r["gold"] for r in rows]
pred = [r["predicted_index"] for r in rows]

valid = [(g, p) for g, p in zip(gold, pred) if isinstance(p, int) and p >= 0]
gold_v = [g for g, _ in valid]
pred_v = [p for _, p in valid]

def accuracy(y_true, y_pred):
    return sum(int(a == b) for a, b in zip(y_true, y_pred)) / max(1, len(y_true))

def macro_f1(y_true, y_pred, classes):
    f1s = []
    for c in classes:
        tp = sum((yt == c and yp == c) for yt, yp in zip(y_true, y_pred))
        fp = sum((yt != c and yp == c) for yt, yp in zip(y_true, y_pred))
        fn = sum((yt == c and yp != c) for yt, yp in zip(y_true, y_pred))
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1s.append(f1)
    return sum(f1s) / len(f1s) if f1s else None

classes = sorted(set(gold))  # usually [0,1,2]
cm = [[0 for _ in classes] for __ in classes]
for g, p in valid:
    cm[g][p] += 1

metrics = {
    "run_id": run_id,
    "model": MODEL_NAME,
    "dataset": FPB_REPO,
    "split": SPLIT,
    "ds_labels": DS_LABELS,
    "n_total": len(rows),
    "n_valid": len(valid),
    "accuracy_all": accuracy(gold, pred),
    "accuracy_valid": accuracy(gold_v, pred_v) if valid else None,
    "macro_f1_valid": macro_f1(gold_v, pred_v, classes) if valid else None,
    "confusion_matrix_classes": classes,
    "confusion_matrix": cm,
}

print(metrics)

out_path = f"evaluation/{run_id}_metrics.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("✅ Metrics saved:", out_path)


{'run_id': 'llama31_8b_fpb_ChanceFocus_en-fpb_test_DSLABELS', 'model': 'meta-llama/Llama-3.1-8B-Instruct', 'dataset': 'ChanceFocus/en-fpb', 'split': 'test', 'ds_labels': ['negative', 'neutral', 'positive'], 'n_total': 970, 'n_valid': 968, 'accuracy_all': 0.4536082474226804, 'accuracy_valid': 0.45454545454545453, 'macro_f1_valid': 0.2694193572933461, 'confusion_matrix_classes': [0, 1, 2], 'confusion_matrix': [[2, 59, 216], [14, 438, 123], [90, 26, 0]]}
✅ Metrics saved: evaluation/llama31_8b_fpb_ChanceFocus_en-fpb_test_DSLABELS_metrics.json
